# Mirage Benchmark Analysis

Resource scaling + regression, the regression-derived `modules.config`, and registration accuracy (tiled vs classic). Set `RESULTS_ROOT` etc. below. All logic lives in `benchmarks/analysis/lib` (tested).

In [ ]:
import matplotlib
%matplotlib inline
from pathlib import Path
from benchmarks.analysis.lib import load, regress, emit_config, plotting
from benchmarks.analysis import make_figures
plotting.set_paper_theme()
# EDIT THESE to your sweep outputs:
RESULTS_ROOT = Path('../../bench_results')
RUN_PLAN = Path('../../bench_run_plan.csv')
MANIFEST = Path('../../bench_matrix/matrix_manifest.csv')
REG_EVAL = Path('../../reg_eval.csv')  # from Plan 2; may not exist yet

## 1. Load + tidy

In [ ]:
df = load.load_runs(RESULTS_ROOT, RUN_PLAN, MANIFEST)
print(df.shape)
df.head()

## 2. Resource ~ input-size regression (per process)

In [ ]:
models = regress.fit_per_process(df, predictor='input_gb', target='peak_rss_gb')
for p, m in sorted(models.items()):
    print(f"{p:24s} slope={m['slope']:.2f} intercept={m['intercept']:.2f} r2={m['r2']:.2f} sigma={m['sigma']:.2f} n={m['n']}")

In [ ]:
for proc, g in df.groupby('process'):
    sub = g[['input_gb','peak_rss_gb']].dropna()
    if sub.empty: continue
    m = models[proc]
    fig = plotting.scatter_with_fit(sub['input_gb'], sub['peak_rss_gb'],
        m['slope'], m['intercept'], 'input (GiB)', 'peak RSS (GiB)', proc)
    plotting.save_fig(fig, Path('figures')/f'scaling_{proc}')

## 3. Optimal modules.config

In [ ]:
emit_config.write_optimized_config(models, '../../conf/modules.optimized.config')
print(open('../../conf/modules.optimized.config').read())

## 4. Registration accuracy — tiled vs classic (needs Plan 2 reg_eval.csv)

In [ ]:
import pandas as pd
if REG_EVAL.exists():
    reg = pd.read_csv(REG_EVAL)
    piv = reg.pivot_table(index='pair_id', columns='mode', values='true_median_rtre')
    fig = plotting.before_after_box(piv, cols=list(piv.columns),
        ylabel='median rTRE', title='valis vs STARE')
    plotting.save_fig(fig, Path('figures')/'rtre_valis_vs_stare')
    display(reg.groupby('mode')[['true_median_rtre','valis_rtre']].median())
else:
    print('reg_eval.csv not found - run Plan 2 first')

## 5. Sampling — paired valis-vs-STARE significance

In [ ]:
from benchmarks.registration_eval import sampling
if REG_EVAL.exists():
    reg = pd.read_csv(REG_EVAL)
    piv = reg.pivot_table(index='pair_id', columns='mode', values='true_median_px').dropna()
    if {'valis','tiled'} <= set(piv.columns):
        print(sampling.paired_diff_test(piv['valis'].values, piv['tiled'].values))